In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns
import sys
import glob
import tqdm


plt.style.use('default')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 22, 'legend.facecolor': 'white', 'legend.framealpha': 1, "legend.frameon": 1, "lines.linewidth": 2})

In [2]:
sys.path.append("/extdata4/baeklab/Hyeonseo/m6A/modformer")
from utils import utils

In [3]:
refflat_df = utils.parse_refflat_v2()

In [4]:
print(refflat_df)

                    transcript_id     gene_id   gene_name    chr strand  \
0                 ENST00000000233        ARF5  ARF5,FSCN3   chr7      +   
1                 ENST00000000442       ESRRA       ESRRA  chr11      +   
2                 ENST00000001008       FKBP4       FKBP4  chr12      +   
3                 ENST00000002125     NDUFAF7     NDUFAF7   chr2      +   
4                 ENST00000002165       FUCA2       FUCA2   chr6      -   
...                           ...         ...         ...    ...    ...   
542603  unassigned_transcript_995  TRH-GTG1-5  TRH-GTG1-5   chr6      +   
542604  unassigned_transcript_996  TRT-AGT6-1  TRT-AGT6-1   chr6      +   
542605  unassigned_transcript_997  TRI-AAT5-2  TRI-AAT5-2   chr6      -   
542606  unassigned_transcript_998  TRV-CAC6-1  TRV-CAC6-1   chr6      -   
542607  unassigned_transcript_999  TRS-CGA2-1  TRS-CGA2-1   chr6      +   

          txStart      txEnd   cdsStart     cdsEnd  exonCount  \
0       127588410  127591700  1275

In [5]:
refflat_df = refflat_df.groupby("gene_id")

In [29]:
def merge_exons(group):
    exonstarts = np.concatenate(group["exonStarts"].values)
    exonends = np.concatenate(group["exonEnds"].values)
    exonbounds = np.concatenate([exonstarts, exonends], axis=0)
    exonbounds_argsort = np.argsort(exonbounds)
    exonbounds = exonbounds[exonbounds_argsort]
    brackets = np.concatenate([np.ones_like(exonstarts), -np.ones_like(exonends)], axis=0)[exonbounds_argsort]
    brackets = np.cumsum(brackets) * brackets
    mergedstarts = exonbounds[brackets == 1]
    mergedends = exonbounds[brackets == 0]
    return mergedstarts, mergedends

In [35]:
df_list = []
for gene_id, group in tqdm.tqdm(refflat_df):
    mergedstarts, mergedends = merge_exons(group)
    chromosome = group["chr"].values[0]
    strand = group["strand"].values[0]
    df_list.append((gene_id, chromosome, strand, mergedstarts, mergedends))
merged_df = pd.DataFrame(df_list, columns=["gene_id", "chr", "strand", "exonStarts", "exonEnds"])

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 78969/78969 [00:15<00:00, 5050.13it/s]


In [36]:
print(merged_df)

        gene_id    chr strand  \
0          A1BG  chr19      -   
1      A1BG-AS1  chr19      +   
2          A1CF  chr10      -   
3           A2M  chr12      -   
4       A2M-AS1  chr12      +   
...         ...    ...    ...   
78964    ZYG11A   chr1      +   
78965    ZYG11B   chr1      +   
78966       ZYX   chr7      +   
78967     ZZEF1  chr17      -   
78968      ZZZ3   chr1      -   

                                              exonStarts  \
0      [58345177, 58347352, 58348465, 58349564, 58351...   
1                         [58347717, 58351969, 58353713]   
2      [50799408, 50809893, 50811039, 50813856, 50816...   
3      [9067663, 9068739, 9069744, 9070487, 9072358, ...   
4                                     [9065162, 9065807]   
...                                                  ...   
78964  [52842510, 52854464, 52856997, 52860730, 52863...   
78965  [52726452, 52754432, 52756457, 52771019, 52779...   
78966  [143381294, 143381906, 143382247, 143382800, 1...   
789

In [37]:
merged_df.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/anno/merged_exons.pkl")